# Time-to-platinum: vitals × feature-statistic ablation

Fit **XGBoost survival:cox and elastic-net Cox** on the full ADT cohort at **0, 90, and 180 days**.
At every landmark, cross **with / without vitals** with **mean + n_obs + missing / all statistics**:
24 fits, each with five-fold tuning and held-out test predictions.

All arms retain age, the same prebuilt patient splits, landmark risk sets, death-censored platinum outcome,
and evaluation horizons. Follow-up is measured from each landmark. Only predictor candidates change;
coverage/variability selection, imputation, scaling, and missingness indicators are learned within training folds.

- `mean-n-obs`: `mean` + `n_observations` (the stored name for n_obs), with train-derived missing indicators.
- `all`: every stored summary statistic (currently mean, min, max, last, delta, n_observations), with the same missing-indicator logic.
- Vitals: height, weight, temperature, heart rate, respiratory rate, systolic and diastolic blood pressure,
  using the canonical names printed below. Inclusion makes them eligible; training coverage/variability gates still apply.

Requires the prediction inputs from `01_preprocessing.ipynb` and the modeling environment used by `03_multivariate.ipynb`.
Results go into a separate `platinum_feature_ablation` tree. Existing completed fits resume by default.

In [ ]:
import json
import shlex
import subprocess
import sys
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Resolve imports whether launched from the repository root or notebook directory.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "COMPASS/survival_analysis/multivariate_analysis.py").exists())
sys.path.insert(0, str(REPO_ROOT))
from COMPASS.survival_analysis.cox_aggregated import VITAL_MEASUREMENTS

SURVIVAL_ROOT = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis")
INPUTS_DIR = SURVIVAL_ROOT / "prediction_inputs_adt"
OUTPUT_ROOT = SURVIVAL_ROOT / "local_runs_adt" / "platinum_feature_ablation"
LANDMARKS = [0, 90, 180]
MODELS = ["elastic-net", "xgboost"]
VITAL_SETTINGS = ["with_vitals", "without_vitals"]
STAT_SETTINGS = ["mean-n-obs", "all"]
N_FOLDS = 5
SEED = 42
OVERWRITE = False
DRY_RUN = False  # print commands without fitting or writing outputs
OUT_OF_FOLD_RISKS = False  # optional nested CV; test-block predictions are always written

TASKS = []
for model, landmark, vitals, stats in product(MODELS, LANDMARKS, VITAL_SETTINGS, STAT_SETTINGS):
    folder = OUTPUT_ROOT / vitals / stats / model / f"landmark_{landmark}"
    prefix = "cox_agg_multivariable" if model == "elastic-net" else "landmark_xgboost"
    cmd = [sys.executable, str(REPO_ROOT / "COMPASS/survival_analysis/multivariate_analysis.py"),
           "--model", model, "--inputs-dir", str(INPUTS_DIR.resolve()), "--output-dir", str(folder.resolve()),
           "--endpoints", "platinum", "--landmark-days", str(landmark), "--feature-set", "labs",
           "--feature-stats", stats, "--n-folds", str(N_FOLDS), "--seed", str(SEED), "--cohort", "all",
           "--overwrite" if OVERWRITE else "--no-overwrite"]
    if vitals == "without_vitals":
        cmd.append("--exclude-vitals")
    if OUT_OF_FOLD_RISKS:
        cmd.append("--out-of-fold-risks")
    TASKS.append(dict(model=model, landmark=landmark, vitals=vitals, stats=stats,
                      folder=folder, metrics=folder / f"{prefix}_metrics_landmark{landmark}.csv",
                      command=cmd))
print(f"{len(TASKS)} fits; vitals: {', '.join(sorted(VITAL_MEASUREMENTS))}")
display(pd.DataFrame(TASKS)[["model", "landmark", "vitals", "stats", "folder"]])

## Check inputs

This checks all landmarks before starting any fit and shows which stored measurements/statistics are available.
No patient is excluded for missing vitals. A missing vital panel raises an error so the two vitals arms cannot
silently run on identical feature sets. The source prediction inputs are read directly and never rewritten.

In [ ]:
required = [INPUTS_DIR / "build_manifest.json"]
for lm in LANDMARKS:
    required.extend([INPUTS_DIR / f"aggregated_landmark{lm}.csv",
                     INPUTS_DIR / f"pre_treatment_lab_long_landmark{lm}.csv"])
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Build inputs with notebook 01 or update INPUTS_DIR:\n" + "\n".join(missing))
build_manifest = json.loads((INPUTS_DIR / "build_manifest.json").read_text())
audit = []
for lm in LANDMARKS:
    cols = pd.read_csv(INPUTS_DIR / f"aggregated_landmark{lm}.csv", nrows=0).columns
    features = [c for c in cols if "__" in c]
    names = {c.rsplit("__", 1)[0] for c in features}
    present_vitals = names & VITAL_MEASUREMENTS
    if not present_vitals:
        raise ValueError(f"No canonical vitals found at landmark {lm}; rebuild inputs including vitals.")
    if not build_manifest["auc_horizons_by_landmark"][str(lm)]["platinum"]:
        raise ValueError(f"No platinum evaluation horizons for landmark {lm}.")
    for vitals, stats in product(VITAL_SETTINGS, STAT_SETTINGS):
        selected = [c for c in features
                    if (vitals == "with_vitals" or c.rsplit("__", 1)[0] not in VITAL_MEASUREMENTS)
                    and (stats == "all" or c.rsplit("__", 1)[1] in {"mean", "n_observations"})]
        if not selected:
            raise ValueError(f"No candidates for {lm}, {vitals}, {stats}")
        audit.append(dict(landmark=lm, vitals=vitals, stats=stats, n_candidates=len(selected),
                          stored_stats=sorted({c.rsplit("__", 1)[1] for c in selected}),
                          available_vitals=sorted(present_vitals)))
display(pd.DataFrame(audit))

## Fit all variants

Each task writes the usual metrics, per-patient held-out risk scores, feature selections, and coefficients or
importance files. `run_settings.json` records the ablation and command. Changing settings in an existing output
folder requires `OVERWRITE = True` or a new `OUTPUT_ROOT`. To refit after changing source data or model code,
set `OVERWRITE = True` as well. Nested out-of-fold predictions are optional because they add substantial tuning work.

In [ ]:
for task in TASKS:
    print(shlex.join(task["command"]), flush=True)
    if DRY_RUN:
        continue
    folder = task["folder"]
    settings = {k: task[k] for k in ("model", "landmark", "vitals", "stats")}
    settings.update(endpoint="platinum", inputs_dir=str(INPUTS_DIR.resolve()), n_folds=N_FOLDS,
                    seed=SEED, out_of_fold_risks=OUT_OF_FOLD_RISKS, build_manifest=build_manifest)
    settings_path = folder / "run_settings.json"
    if settings_path.exists() and not OVERWRITE:
        previous = json.loads(settings_path.read_text())
        if previous["settings"] != settings:
            raise ValueError(f"Settings changed under {folder}; use OVERWRITE=True or a new OUTPUT_ROOT.")
    if task["metrics"].exists() and not OVERWRITE:
        if not settings_path.exists():
            raise ValueError(f"Cannot verify existing fit under {folder}; use OVERWRITE=True.")
        print(f"[skip] {task['metrics']}")
        continue
    folder.mkdir(parents=True, exist_ok=True)
    settings_path.write_text(json.dumps({"settings": settings, "command": task["command"]}, indent=2) + "\n")
    subprocess.run(task["command"], cwd=REPO_ROOT, check=True)

## Held-out comparison

Compare mean AUC(t), C-index, and integrated Brier on each landmark's test set. Higher AUC/C-index and lower
Brier are better. Different landmarks have different at-risk populations, so changes across landmarks do not
isolate the benefit of accumulating history. The following contrasts hold model and landmark fixed.

In [ ]:
rows = []
for task in TASKS:
    row = {k: task[k] for k in ("model", "landmark", "vitals", "stats")}
    row.update(status="missing", source_path=str(task["metrics"]))
    if task["metrics"].exists():
        frame = pd.read_csv(task["metrics"])
        frame = frame.loc[frame["endpoint"] == "platinum"]
        if len(frame) != 1:
            raise ValueError(f"Expected one platinum metrics row: {task['metrics']}")
        row.update(frame.iloc[0].to_dict())
        row.update({k: task[k] for k in ("model", "landmark", "vitals", "stats")})
        row["status"] = "ok"
    rows.append(row)
summary = pd.DataFrame(rows)
metrics = ["test_mean_auc_t", "test_c_index", "test_integrated_brier"]
for col in [*metrics, "n_test", "n_events_test"]:
    if col not in summary:
        summary[col] = np.nan
display(summary[["model", "landmark", "vitals", "stats", "status", "n_test", "n_events_test", *metrics]])
if not DRY_RUN:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    summary.to_csv(OUTPUT_ROOT / "ablation_summary.csv", index=False)

ok = summary.loc[summary["status"] == "ok"]
if not ok.empty:
    counts = ok.groupby("landmark")[["n_test", "n_events_test"]].nunique()
    if (counts > 1).any().any():
        raise ValueError("Test cohort counts differ across variants; inspect inputs and stale outputs before comparing.")
    display(ok.pivot(index=["model", "vitals", "stats"], columns="landmark", values=metrics).round(3))

In [ ]:
# Paired experimental contrasts; point estimates, without confidence intervals.
contrasts = []
for axis, other, positive, negative in [
    ("vitals", "stats", "with_vitals", "without_vitals"),
    ("stats", "vitals", "all", "mean-n-obs"),
]:
    keys = ["model", "landmark", other]
    pairs = ok.loc[ok[axis] == positive].merge(ok.loc[ok[axis] == negative], on=keys,
                                             suffixes=("_positive", "_negative"), validate="one_to_one")
    for _, pair in pairs.iterrows():
        contrast = {k: pair[k] for k in keys}
        contrast["contrast"] = f"{positive} minus {negative}"
        for metric in metrics:
            contrast[f"delta_{metric}"] = pair[f"{metric}_positive"] - pair[f"{metric}_negative"]
        contrasts.append(contrast)
contrast_df = pd.DataFrame(contrasts)
display(contrast_df)
if not DRY_RUN:
    contrast_df.to_csv(OUTPUT_ROOT / "ablation_contrasts.csv", index=False)

if not ok.empty:
    fig, axes = plt.subplots(1, len(LANDMARKS), figsize=(6 * len(LANDMARKS), 5), sharey=True, squeeze=False)
    for ax, lm in zip(axes[0], LANDMARKS):
        sub = ok.loc[ok["landmark"] == lm].copy()
        sub["label"] = sub["model"] + " | " + sub["vitals"] + " | " + sub["stats"]
        ax.barh(sub["label"], sub["test_mean_auc_t"])
        ax.axvline(0.5, color="black", ls=":", lw=0.7)
        ax.set_xlim(0, 1)
        ax.set_title(f"Landmark {lm} days")
        ax.set_xlabel("Test mean AUC(t)")
    fig.tight_layout()
    if not DRY_RUN:
        fig.savefig(OUTPUT_ROOT / "ablation_mean_auc.png", dpi=180, bbox_inches="tight")
    plt.show()